The final pipeline is src/train.py, which produced validation_predictions.csv and the December chart. Small differences in December values come from the script's extra safeguards.

In [27]:
import numpy as np, pandas as pd

train = pd.read_csv("data/train_test.csv", parse_dates=["date"])
val   = pd.read_csv("data/validation.csv", parse_dates=["date"])

def haversine(lat1, lon1, lat2, lon2):
    a, b, c, d = map(np.radians, [lat1, lon1, lat2, lon2])
    x = np.sin((c-a)/2)**2 + np.cos(a)*np.cos(c)*np.sin((d-b)/2)**2
    return 3958.8 * 2 * np.arcsin(np.sqrt(x))

print(train.shape, val.shape)

(48000, 14) (12000, 13)


In [28]:
import numpy as np, pandas as pd

train = pd.read_csv("data/train_test.csv", parse_dates=["date"])
val   = pd.read_csv("data/validation.csv", parse_dates=["date"])

print(train.shape, val.shape)
print(train.date.min(), train.date.max())
print(val.date.min(), val.date.max())
train.head()

(48000, 14) (12000, 13)
2025-01-01 00:00:00 2025-10-31 00:00:00
2025-11-01 00:00:00 2025-12-31 00:00:00


,load_id,pickup,delivery,pickup_lat,pickup_lon,delivery_lat,delivery_lon,distance,equipment,weight,date,market_index,quote_signal,posted_rate
0,TR-000001,Richmond,Baltimore,38.09122,-76.78906,38.16908,-72.74564,274.3,Dry Van,30658.0,2025-01-01,0.95684,2.39595,645.41
1,TR-000002,Richmond,Philadelphia,38.09122,-76.78906,39.22317,-72.96710,280.5,Reefer,17555.0,2025-01-01,0.97623,2.43355,679.97
2,TR-000003,Philadelphia,Green Bay,39.22317,-72.96710,44.30296,-87.52871,967.8,Dry Van,31721.0,2025-01-01,1.00971,1.84491,1802.54
3,TR-000004,Hartford,Atlanta,39.55328,-72.18051,34.84933,-86.28940,965.4,Dry Van,32333.0,2025-01-01,0.94518,1.87712,1827.28
4,TR-000005,Dallas,Nashville,31.83025,-94.38343,35.29479,-88.08915,541.9,Reefer,35183.0,2025-01-01,0.98480,2.56300,1380.28


In [29]:
print(train.isna().sum())
print()
print(val.isna().sum())
train[["distance", "weight", "market_index", "quote_signal", "posted_rate"]].describe()

load_id           0
pickup            0
delivery          0
pickup_lat        0
pickup_lon        0
delivery_lat      0
delivery_lon      0
distance          0
equipment         0
weight          300
date              0
market_index    374
quote_signal      0
posted_rate       0
dtype: int64

load_id           0
pickup            0
delivery          0
pickup_lat        0
pickup_lon        0
delivery_lat      0
delivery_lon      0
distance          0
equipment         0
weight          165
date              0
market_index    249
quote_signal      0
dtype: int64


,distance,weight,market_index,quote_signal,posted_rate
count,48000.000000,47700.000000,47626.000000,48000.000000,48000.000000
mean,1135.856654,31028.844004,1.083387,2.062468,2373.980682
std,728.564416,9391.440620,0.168091,0.291391,1486.493245
min,70.000000,-47500.000000,0.676390,0.692280,57.220000
25%,550.400000,25800.000000,0.949670,1.891030,1251.555000
50%,953.300000,31436.500000,1.055800,2.055750,2030.760000
75%,1645.525000,37018.000000,1.219590,2.221685,3330.750000
max,3439.800000,47500.000000,1.467780,3.610350,25533.000000


In [30]:
neg = train[train.weight < 0]
print("negative weights:", len(neg))
print(train.weight.abs().describe())

negative weights: 292
count    47700.000000
mean     31417.249245
std       7996.515399
min       5000.000000
25%      25922.000000
50%      31496.000000
75%      37063.250000
max      47500.000000
Name: weight, dtype: float64


In [31]:
train["rpm"] = train.posted_rate / train.distance
print(train.rpm.describe())
print("above $5/mile:", (train.rpm > 5).sum())
print("below $1/mile:", (train.rpm < 1).sum())

count    48000.000000
mean         2.215347
std          0.583887
min          0.333709
25%          1.986974
50%          2.145348
75%          2.342696
max         14.125294
Name: rpm, dtype: float64
above $5/mile: 317
below $1/mile: 329


In [32]:
def haversine(lat1, lon1, lat2, lon2):
    a, b, c, d = map(np.radians, [lat1, lon1, lat2, lon2])
    x = np.sin((c-a)/2)**2 + np.cos(a)*np.cos(c)*np.sin((d-b)/2)**2
    return 3958.8 * 2 * np.arcsin(np.sqrt(x))

train["hav"] = haversine(train.pickup_lat, train.pickup_lon, train.delivery_lat, train.delivery_lon)
train["ratio"] = train.distance / train.hav
print(train.ratio.describe())
print("suspicious distances:", ((train.ratio > 1.6) | (train.ratio < 1.0)).sum())

count    48000.000000
mean         1.194681
std          0.143541
min          1.058090
25%          1.164765
50%          1.182231
75%          1.204765
max          9.634919
Name: ratio, dtype: float64
suspicious distances: 111


In [33]:
new_cities = (set(val.pickup) | set(val.delivery)) - (set(train.pickup) | set(train.delivery))
print(new_cities)
share = (val.pickup.isin(new_cities) | val.delivery.isin(new_cities)).mean()
print("share of validation rows touching them:", round(share, 3))

{'San Diego', 'Chicago', 'Laredo', 'Allentown', 'Charlotte', 'Norfolk', 'Jackson', 'Knoxville'}
share of validation rows touching them: 0.121


In [34]:
print(train.groupby(train.date.dt.month)[["posted_rate", "market_index", "quote_signal"]].mean().round(2))
print(val.groupby(val.date.dt.month)[["market_index", "quote_signal"]].mean().round(2))

      posted_rate  market_index  quote_signal
date                                         
1         2255.97          0.93          2.07
2         2273.80          1.00          2.10
3         2372.27          1.07          2.18
4         2372.16          1.21          1.96
5         2421.78          1.30          1.91
6         2497.03          1.28          2.30
7         2415.16          1.20          1.92
8         2338.41          0.98          2.05
9         2406.37          0.89          2.20
10        2379.05          0.96          1.94
      market_index  quote_signal
date                            
11            0.92          2.06
12            0.93          2.05


In [35]:
def clean(d):
    d = d.copy()

    # 1. Weight: negative = sign error, missing = fill with median of same truck type
    d["weight_missing"] = d.weight.isna().astype(int)
    d["weight"] = d.weight.abs()
    d["weight"] = d.weight.fillna(d.groupby("equipment").weight.transform("median"))

    # 2. Distance: road distance should be ~1.18x straight-line; rebuild impossible ones
    d["hav"] = haversine(d.pickup_lat, d.pickup_lon, d.delivery_lat, d.delivery_lon)
    ratio = d.distance / d.hav
    bad = (ratio > 1.6) | (ratio < 1.0)
    d["distance_bad"] = bad.astype(int)
    d["distance_fixed"] = np.where(bad, d.hav * 1.182, d.distance)

    # 3. market_index: fill with the median of loads on the same date
    d["market_missing"] = d.market_index.isna().astype(int)
    d["market_index"] = d.market_index.fillna(d.groupby("date").market_index.transform("median"))
    return d

T = clean(train)
V = clean(val)

print("negative weights left:", (T.weight < 0).sum())
print("missing weight left:", T.weight.isna().sum())
print("missing market_index left:", T.market_index.isna().sum(), V.market_index.isna().sum())
print("distances rebuilt:", T.distance_bad.sum(), V.distance_bad.sum())

negative weights left: 0
missing weight left: 0
missing market_index left: 0 0
distances rebuilt: 111 34


In [36]:
# Date-level market conditions: median market_index and quote_signal for each date.
# Uses train + validation rows (these are inputs, not prices, so no target leakage).
daily = pd.concat([T, V]).groupby("date")[["market_index", "quote_signal"]].median()

def features(d):
    X = pd.DataFrame(index=d.index)

    # Distance: the strongest driver of price
    X["log_dist"] = np.log(d.distance_fixed)
    X["distance"] = d.distance_fixed
    X["hav"] = d.hav

    # Truck type and load
    X["equip"] = d.equipment.map({"Dry Van": 0, "Flatbed": 1, "Reefer": 2})
    X["weight"] = d.weight
    X["weight_missing"] = d.weight_missing

    # Market conditions
    X["market_index"] = d.market_index
    X["market_missing"] = d.market_missing
    X["quote_signal"] = d.quote_signal
    X["qs_dev"] = d.quote_signal - 2.0
    X["qs_absdev"] = (d.quote_signal - 2.0).abs()
    X["day_mkt"] = d.date.map(daily.market_index)
    X["day_qs"] = d.date.map(daily.quote_signal)

    # Day of week (NOT day of year; see below)
    X["dow"] = d.date.dt.dayofweek

    # Location as numbers instead of city names
    for c in ["pickup_lat", "pickup_lon", "delivery_lat", "delivery_lon"]:
        X[c] = d[c]
    X["dlat"] = d.delivery_lat - d.pickup_lat
    X["dlon"] = d.delivery_lon - d.pickup_lon
    return X

XT = features(T)
XV = features(V)
y = T.posted_rate.values

print(XT.shape, XV.shape)
print("missing values:", XT.isna().sum().sum(), XV.isna().sum().sum())
XT.head()

(48000, 20) (12000, 20)
missing values: 0 0


,log_dist,distance,hav,equip,weight,weight_missing,market_index,market_missing,quote_signal,qs_dev,qs_absdev,day_mkt,day_qs,dow,pickup_lat,pickup_lon,delivery_lat,delivery_lon,dlat,dlon
0,5.614222,274.3,219.808768,0,30658.0,0,0.95684,0,2.39595,0.39595,0.39595,0.96259,2.01772,2,38.09122,-76.78906,38.16908,-72.74564,0.07786,4.04342
1,5.636574,280.5,220.525617,2,17555.0,0,0.97623,0,2.43355,0.43355,0.43355,0.96259,2.01772,2,38.09122,-76.78906,39.22317,-72.96710,1.13195,3.82196
2,6.875025,967.8,826.919847,0,31721.0,0,1.00971,0,1.84491,-0.15509,0.15509,0.96259,2.01772,2,39.22317,-72.96710,44.30296,-87.52871,5.07979,-14.56161
3,6.872543,965.4,840.540749,0,32333.0,0,0.94518,0,1.87712,-0.12288,0.12288,0.96259,2.01772,2,39.55328,-72.18051,34.84933,-86.28940,-4.70395,-14.10889
4,6.295081,541.9,434.163187,2,35183.0,0,0.98480,0,2.56300,0.56300,0.56300,0.96259,2.01772,2,31.83025,-94.38343,35.29479,-88.08915,3.46454,6.29428


In [37]:
import lightgbm as lgb

dist = T.distance_fixed.values

is_train = (T.date < "2025-09-01").values   # Jan-Aug: the model learns from these
is_test  = ~is_train                        # Sep-Oct: the model is scored on these

print("train rows:", is_train.sum(), " test rows:", is_test.sum())

train rows: 38477  test rows: 9523


In [38]:
def metrics(yt, yp):
    return {"MAE $": round(np.mean(abs(yt - yp)), 1),
            "MedAE $": round(np.median(abs(yt - yp)), 1),
            "MAPE %": round(np.mean(abs(yt - yp) / yt) * 100, 2)}

bins = [0, 150, 300, 600, 1000, 1500, 2500, 5000]
T["dbin"] = pd.cut(T.distance_fixed, bins)
rpm = T.posted_rate / T.distance_fixed

# typical price per mile for each (truck type, distance band), learned from Jan-Aug only
med = rpm[is_train].groupby([T.equipment[is_train], T.dbin[is_train]], observed=True).median()
keys = list(zip(T.equipment[is_test], T.dbin[is_test]))
base_pred = np.array([med.get(k, 2.1) for k in keys]) * dist[is_test]

print("baseline:", metrics(y[is_test], base_pred))

baseline: {'MAE $': np.float64(138.2), 'MedAE $': np.float64(62.5), 'MAPE %': np.float64(6.36)}


In [39]:
model = lgb.LGBMRegressor(
    objective="l1",            # robust to the extreme-price outliers
    n_estimators=600, learning_rate=0.03,
    num_leaves=63, min_child_samples=20,
    subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
    random_state=0, verbose=-1)

# target = log(price per mile): distance is divided out, then multiplied back
model.fit(XT[is_train], np.log(y[is_train]) - np.log(dist[is_train]))
pred = np.exp(model.predict(XT[is_test]) + np.log(dist[is_test]))

print("model:   ", metrics(y[is_test], pred))

model:    {'MAE $': np.float64(110.4), 'MedAE $': np.float64(36.2), 'MAPE %': np.float64(4.75)}


In [40]:
rng = np.random.RandomState(0)
rand_train = rng.rand(len(T)) < 0.8

m2 = lgb.LGBMRegressor(objective="l1", n_estimators=600, learning_rate=0.03,
                       num_leaves=63, min_child_samples=20, subsample=0.8,
                       subsample_freq=1, colsample_bytree=0.8, random_state=0, verbose=-1)
m2.fit(XT[rand_train], np.log(y[rand_train]) - np.log(dist[rand_train]))
p2 = np.exp(m2.predict(XT[~rand_train]) + np.log(dist[~rand_train]))

print("random split:", metrics(y[~rand_train], p2))

random split: {'MAE $': np.float64(78.3), 'MedAE $': np.float64(16.4), 'MAPE %': np.float64(3.77)}


In [41]:
# Final model: train on ALL 48,000 labeled loads, average 5 seeds for stability
models = []
for seed in range(5):
    m = lgb.LGBMRegressor(objective="l1", n_estimators=600, learning_rate=0.03,
                          num_leaves=63, min_child_samples=20, subsample=0.8,
                          subsample_freq=1, colsample_bytree=0.8,
                          random_state=seed, verbose=-1)
    m.fit(XT, np.log(y) - np.log(dist))
    models.append(m)

def predict(X, distance):
    return np.exp(np.mean([m.predict(X) for m in models], axis=0) + np.log(distance))

val_pred = predict(XV, V.distance_fixed.values)

template = pd.read_csv("data/validation_predictions_template.csv")
assert list(template.load_id) == list(val.load_id), "load_id order differs from template!"

out = pd.DataFrame({"load_id": val.load_id, "predicted_rate": val_pred.round(2)})
out.to_csv("validation_predictions.csv", index=False)

print(out.shape, out.predicted_rate.min(), out.predicted_rate.max(), out.predicted_rate.isna().sum())
out.head()

(12000, 2) 2.95 6676.96 0


,load_id,predicted_rate
0,TE-000001,819.19
1,TE-000002,4815.57
2,TE-000003,5241.42
3,TE-000004,4134.82
4,TE-000005,1790.86


In [42]:
dec = pd.read_csv("data/december_chart_inputs.csv", parse_dates=["date"])
original_cols = list(dec.columns)

# look up coordinates for the two cities from the training data
coords = pd.concat([
    train[["pickup", "pickup_lat", "pickup_lon"]].set_axis(["city", "lat", "lon"], axis=1),
    train[["delivery", "delivery_lat", "delivery_lon"]].set_axis(["city", "lat", "lon"], axis=1),
]).drop_duplicates("city").set_index("city")

D = dec.copy()
D["pickup_lat"]    = D.pickup.map(coords.lat);    D["pickup_lon"]    = D.pickup.map(coords.lon)
D["delivery_lat"]  = D.delivery.map(coords.lat);  D["delivery_lon"]  = D.delivery.map(coords.lon)
D["market_index"]  = D.date.map(daily.market_index)   # per-date market values from validation
D["quote_signal"]  = D.date.map(daily.quote_signal)

D = clean(D)
D["distance_fixed"] = D.distance        # the 360 miles is given, so keep it
XD = features(D)

dec["predicted_rate"] = predict(XD, D.distance_fixed.values).round(2)
dec[original_cols].to_csv("data/december_chart_inputs.csv", index=False, date_format="%Y-%m-%d")

print(dec.predicted_rate.describe())
dec.head()

count     31.000000
mean     788.478710
std        5.647082
min      777.170000
25%      785.425000
50%      787.710000
75%      792.010000
max      801.520000
Name: predicted_rate, dtype: float64


,pickup,delivery,distance,equipment,weight,date,predicted_rate
0,Lexington,Fort Wayne,360,Dry Van,32000,2025-12-01,782.67
1,Lexington,Fort Wayne,360,Dry Van,32000,2025-12-02,783.79
2,Lexington,Fort Wayne,360,Dry Van,32000,2025-12-03,786.74
3,Lexington,Fort Wayne,360,Dry Van,32000,2025-12-04,797.94
4,Lexington,Fort Wayne,360,Dry Van,32000,2025-12-05,792.70
